In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install transformers datasets -q

import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
import pickle
import os

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch version: 2.10.0+cu128
GPU available: True
Device: Tesla T4


In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/intent-chatbot/data/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/intent-chatbot/data/val.csv')
test_df = pd.read_csv('/content/drive/MyDrive/intent-chatbot/data/test.csv')

with open('/content/drive/MyDrive/intent-chatbot/data/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

NUM_CLASSES = len(le.classes_)
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Number of classes:", NUM_CLASSES)

Train: 15797
Val: 1756
Number of classes: 227


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class IntentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IntentDataset(train_df, tokenizer)
val_dataset = IntentDataset(val_df, tokenizer)
test_dataset = IntentDataset(test_df, tokenizer)

print("Datasets created!")
print("Train batches:", len(train_dataset))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Datasets created!
Train batches: 15797


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("DataLoaders created!")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

DataLoaders created!
Train batches: 494
Val batches: 55


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=NUM_CLASSES
)
model = model.to(device)
print("Model loaded on:", device)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: cuda
Total parameters: 109656803


In [ ]:
EPOCHS = 5
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print("Total training steps:", total_steps)
print("Warmup steps:", total_steps // 10)

Total training steps: 2470
Warmup steps: 247


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1

print("Training functions ready!")

Training functions ready!


In [ ]:
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f"✅ Best model saved! Val Acc: {val_acc:.4f}")

print("\nTraining complete!")
print("Best Validation Accuracy:", best_val_acc)


Epoch 1/5
----------------------------------------
Train Loss: 4.9921 | Train Acc: 0.0796
Val Loss:   3.9946 | Val Acc:   0.3007 | Val F1: 0.2430
✅ Best model saved! Val Acc: 0.3007

Epoch 2/5
----------------------------------------
Train Loss: 3.4049 | Train Acc: 0.4963
Val Loss:   2.6930 | Val Acc:   0.6834 | Val F1: 0.6431
✅ Best model saved! Val Acc: 0.6834

Epoch 3/5
----------------------------------------
Train Loss: 2.4050 | Train Acc: 0.7380
Val Loss:   1.9794 | Val Acc:   0.7887 | Val F1: 0.7622
✅ Best model saved! Val Acc: 0.7887

Epoch 4/5
----------------------------------------
Train Loss: 1.8526 | Train Acc: 0.8274
Val Loss:   1.6271 | Val Acc:   0.8314 | Val F1: 0.8129
✅ Best model saved! Val Acc: 0.8314

Epoch 5/5
----------------------------------------
Train Loss: 1.5972 | Train Acc: 0.8635
Val Loss:   1.5164 | Val Acc:   0.8440 | Val F1: 0.8294
✅ Best model saved! Val Acc: 0.8440

Training complete!
Best Validation Accuracy: 0.8439635535307517


In [ ]:
# Load best weights
model.load_state_dict(torch.load('/content/best_model.pt'))

# Save model + tokenizer to Drive
save_path = '/content/drive/MyDrive/intent-chatbot/models/bert-intent-model'
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to Drive!")
print("Path:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive!
Path: /content/drive/MyDrive/intent-chatbot/models/bert-intent-model


In [13]:
# Continue training for 2 more epochs
EXTRA_EPOCHS = 2

for epoch in range(EXTRA_EPOCHS):
    print(f"\nExtra Epoch {epoch+1}/{EXTRA_EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f"✅ Best model saved! Val Acc: {val_acc:.4f}")

print("\nFinal Best Accuracy:", best_val_acc)


Extra Epoch 1/2
----------------------------------------
Train Loss: 1.5478 | Train Acc: 0.8739
Val Loss:   1.5164 | Val Acc:   0.8440 | Val F1: 0.8294

Extra Epoch 2/2
----------------------------------------
Train Loss: 1.5464 | Train Acc: 0.8725
Val Loss:   1.5164 | Val Acc:   0.8440 | Val F1: 0.8294

Final Best Accuracy: 0.8439635535307517


In [14]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

NEW_LR = 5e-6
EXTRA = 3

optimizer2 = AdamW(model.parameters(), lr=NEW_LR, weight_decay=0.01)
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * EXTRA
)

for epoch in range(EXTRA):
    print(f"\nFine-tune Epoch {epoch+1}/{EXTRA}")
    print("-" * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer2, scheduler2, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f"✅ New best saved! Val Acc: {val_acc:.4f}")

print("\nFinal Best Accuracy:", best_val_acc)


Fine-tune Epoch 1/3
----------------------------------------
Train Loss: 1.4667 | Train Acc: 0.8755
Val Loss:   1.3510 | Val Acc:   0.8622 | Val F1: 0.8506
✅ New best saved! Val Acc: 0.8622

Fine-tune Epoch 2/3
----------------------------------------
Train Loss: 1.3092 | Train Acc: 0.8935
Val Loss:   1.2506 | Val Acc:   0.8696 | Val F1: 0.8592
✅ New best saved! Val Acc: 0.8696

Fine-tune Epoch 3/3
----------------------------------------
Train Loss: 1.2297 | Train Acc: 0.9059
Val Loss:   1.2164 | Val Acc:   0.8713 | Val F1: 0.8615
✅ New best saved! Val Acc: 0.8713

Final Best Accuracy: 0.8712984054669703


In [15]:
EXTRA2 = 2

optimizer3 = AdamW(model.parameters(), lr=2e-6, weight_decay=0.01)
scheduler3 = get_linear_schedule_with_warmup(
    optimizer3,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * EXTRA2
)

for epoch in range(EXTRA2):
    print(f"\nFine-tune Epoch {epoch+1}/{EXTRA2}")
    print("-" * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer3, scheduler3, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f"✅ New best saved! Val Acc: {val_acc:.4f}")

print("\nFinal Best Accuracy:", best_val_acc)


Fine-tune Epoch 1/2
----------------------------------------
Train Loss: 1.1878 | Train Acc: 0.9106
Val Loss:   1.1644 | Val Acc:   0.8736 | Val F1: 0.8633
✅ New best saved! Val Acc: 0.8736

Fine-tune Epoch 2/2
----------------------------------------
Train Loss: 1.1406 | Train Acc: 0.9152
Val Loss:   1.1449 | Val Acc:   0.8770 | Val F1: 0.8687
✅ New best saved! Val Acc: 0.8770

Final Best Accuracy: 0.876993166287016


In [16]:
EXTRA3 = 5

optimizer4 = AdamW(model.parameters(), lr=5e-7, weight_decay=0.01)
scheduler4 = get_linear_schedule_with_warmup(
    optimizer4,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * EXTRA3
)

for epoch in range(EXTRA3):
    print(f"\nFinal Push Epoch {epoch+1}/{EXTRA3}")
    print("-" * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer4, scheduler4, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_model.pt')
        print(f"✅ New best saved! Val Acc: {val_acc:.4f}")
    else:
        print(f"No improvement. Best still: {best_val_acc:.4f}")

print("\n🏁 Final Best Accuracy:", best_val_acc)


Final Push Epoch 1/5
----------------------------------------
Train Loss: 1.1264 | Train Acc: 0.9161
Val Loss:   1.1294 | Val Acc:   0.8787 | Val F1: 0.8705
✅ New best saved! Val Acc: 0.8787

Final Push Epoch 2/5
----------------------------------------
Train Loss: 1.1087 | Train Acc: 0.9185
Val Loss:   1.1167 | Val Acc:   0.8793 | Val F1: 0.8708
✅ New best saved! Val Acc: 0.8793

Final Push Epoch 3/5
----------------------------------------
Train Loss: 1.0967 | Train Acc: 0.9183
Val Loss:   1.1085 | Val Acc:   0.8793 | Val F1: 0.8707
No improvement. Best still: 0.8793

Final Push Epoch 4/5
----------------------------------------
Train Loss: 1.0883 | Train Acc: 0.9220
Val Loss:   1.1030 | Val Acc:   0.8804 | Val F1: 0.8725
✅ New best saved! Val Acc: 0.8804

Final Push Epoch 5/5
----------------------------------------
Train Loss: 1.0858 | Train Acc: 0.9195
Val Loss:   1.1014 | Val Acc:   0.8804 | Val F1: 0.8725
No improvement. Best still: 0.8804

🏁 Final Best Accuracy: 0.880410022779

In [17]:
import os

# Load best weights
model.load_state_dict(torch.load('/content/best_model.pt'))

# Save model + tokenizer to Drive
save_path = '/content/drive/MyDrive/intent-chatbot/models/bert-intent-model'
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Model saved to Drive!")
print("📁 Path:", save_path)

# Verify files saved
files = os.listdir(save_path)
print("\nSaved files:")
for f in files:
    size = os.path.getsize(os.path.join(save_path, f)) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Drive!
📁 Path: /content/drive/MyDrive/intent-chatbot/models/bert-intent-model

Saved files:
  config.json — 0.0 MB
  model.safetensors — 418.3 MB
  tokenizer_config.json — 0.0 MB
  tokenizer.json — 0.7 MB
